# Challenge: integration of a singular function

In [ ]:
#    APM41012EP course notebook - Chapter 3 - M. Massot 2026-2027 - École polytechnique
#    ----------   
#    Challenge: integration of a singular function
#    Authors: L. Séries and M. Massot - (C) 2026

In [ ]:
import numpy as np
import plotly.graph_objs as go
import plotly.io as pio
pio.templates.default = "seaborn"
import warnings
warnings.filterwarnings('ignore')

We seek to integrate the function $f(x) = \sqrt(x) \log(x)$ on the interval $]0,1]$.

In [ ]:
def f(x):
    return np.sqrt(x)*np.log(x)

In [ ]:
x = np.linspace(1.e-20, 1, 500)

fig = go.Figure(go.Scatter(x=x, y=f(x), fill='tozeroy', name='f(x)'))
fig.update_layout(title="Integration of the function f(x) = √x log(x)")
fig.show(height=500)

What is the best strategy, in terms of cost, for integrating this function so as to obtain a good accuracy (better than 1e-12): increasing the order of a quadrature formula, or fixing the order of the quadrature and increasing the number of subdivisions? 

To answer the previous question, we have integrated the function $f(x)$ for different numbers of quadrature points of the Clenshaw-Curtis quadrature formula and for subdivisions into different numbers of sub-intervals.

In [ ]:
def cheb_points(n):
    return np.cos( (2*np.arange(0,n)+1)*np.pi / (2*n) )

def coeffs_clenshawcurtis(n):
    x = cheb_points(n+1) # n+1 Chebyshev nodes
    k = np.arange(0, n+1, dtype='float')
    b = np.zeros(n+1)
    b[0::2] = 2 / (1 - k[0::2]*k[0::2])
    theta = (2*np.arange(0,n+1)+1)/(2*(n+1))*np.pi 
    ## Construction of the matrix B from the notes
    B = np.cos(np.outer(np.arange(0,n+1),theta))/(n+1)
    B[1:,:] = 2*B[1:,:]
    w = np.dot(np.transpose(B), b)
    return x, w

In [ ]:
res_exa = -4./9.

# results obtained with adaptive integration
eval_fcn_adapt = 615
err_adapt = 5.2130522121e-13/-res_exa
#err_adapt = 1.9056978218e-13/-res_exa

xmin = 0.
xmax = 1.

nb_intervals = np.array([1, 10, 100, 1000])
nb_pts = np.array([2, 5, 50, 500, 5000])

res = np.zeros((nb_pts.size, nb_intervals.size))
err = np.zeros((nb_pts.size, nb_intervals.size))

for i_n, n in enumerate(nb_pts):
    xk, wk = coeffs_clenshawcurtis(n-1)
    for i_m, m in enumerate(nb_intervals):
        x = np.linspace(xmin, xmax, m+1)
        for j in range(m):
            xminj = x[j]
            xmaxj = x[j+1]
            xkj = xminj + ((xmaxj-xminj)/2)*(xk+1) 
            wkj = ((xmaxj-xminj)/2)*wk
            res[i_n, i_m] += np.sum(wkj*f(xkj))
        err[i_n, i_m] = np.abs(res[i_n, i_m]-res_exa) /  np.abs(res_exa)

**Error as a function of the number of quadrature points**

For subdivisions into different numbers of sub-intervals, we plot the error as a function of the number of quadrature points:

In [ ]:
fig01 = go.Figure()
for i_m, m in enumerate(nb_intervals):
    fig01.add_trace(go.Scatter(x=nb_pts, y=err[:,i_m], mode="markers", name=f"m={m}"))
fig01.update_xaxes(type="log", exponentformat='e', title_text='Nb. of quadrature points per interval')
fig01.update_yaxes(type="log", exponentformat='e', title_text='Error')
fig01.update_layout(height=500, title="Error as a function of the number of quadrature points", legend_title_text="Nb. of intervals")
fig01.show()

**Error as a function of the number of intervals**

For different numbers of quadrature points $n$, we plot the error as a function of the number of intervals:

In [ ]:
fig02 = go.Figure()
for i_n, n in enumerate(nb_pts):
    fig02.add_trace(go.Scatter(x=nb_intervals, y=err[i_n], mode="markers", name=f"n={n}"))
fig02.update_xaxes(type="log", exponentformat='e', title_text="Nb. of intervals")
fig02.update_yaxes(type="log", exponentformat='e', title_text='Error')
fig02.update_layout(height=500, title="Error as a function of the number of intervals", legend_title_text='Nb. of quadrature points')
fig02.show()

**Error as a function of the number of function evaluations**

In [ ]:
fig03 = go.Figure()
for i_n, n in enumerate(nb_pts):
    fig03.add_trace(go.Scatter(x=n*nb_intervals, y=err[i_n], mode="markers", name=f"n={n}"))
fig03.add_trace(go.Scatter(x=[eval_fcn_adapt], y=[err_adapt], mode="markers", marker=dict(size=10), name=f"adaptive<br>integration"))
fig03.update_xaxes(type="log", exponentformat='e', title_text="Nb. of function evaluations")
fig03.update_yaxes(type="log", exponentformat='e', title_text='Error')
fig03.update_layout(height=500, title="Error as a function of the number of function evaluations", legend_title_text='Nb. of quadrature points')
fig03.show()     

The reader may observe, on the plot of the error as a function of the number of function evaluations, that the most efficient strategy for a given accuracy is to take a single interval and to take advantage of the excellent stability of our implementation of the Clenshaw-Curtis method in order to increase the number of quadrature points (the leftmost point, which corresponds to a minimal computational effort for a fixed accuracy, is the one associated with an elementary quadrature).

We are in a situation where the loss of regularity of the function near zero invalidates the convergence theorems, be it in terms of the number of sub-intervals of the subdivision in composite quadratures at a fixed number of quadrature points (plot of the error as a function of the number of intervals), or, for a given number of subdivisions, in terms of the number of quadrature points (plot of the error as a function of the number of quadrature points). Consequently, the latter plot shows that, in order to reach a fixed accuracy better than 1.e-13, one has to perform at least an elementary quadrature with 50000 points, and if one sticks to 5000 quadrature points, then about a hundred intervals are needed, which inevitably leads to a number of function evaluations of the order of 50 000 to 500 000.

The reader may observe that, for this particular function, an adaptive quadrature makes it possible to divide the number of function evaluations by a factor of the order of 100, which corresponds to a very clear speed-up of the computation.

For the adaptive method, at each iteration two order-15 quadratures are computed, which corresponds to 30 function evaluations. For this case, the algorithm converged in 20 iterations to reach an accuracy of the order of 1e-12 (to which one has to add the computation of two additional quadratures during the initialization), which makes 20x30 + 15, that is 615 function evaluations. Strictly speaking, in order to evaluate the cost of the adaptive method, one would have to take into account the computation of the error estimate.

**Remark on the importance of the singularity**

We now perform the integration on the interval $[0.02, 1]$, so as to exclude the singularity at 0.

In [ ]:
xmin = 0.02
xmax = 1.

res_exa = -(4./9.)*(1-xmin**(3/2))-(2/3)*xmin**(3/2)*np.log(xmin)

nb_intervals = np.array([1, 10, 100, 1000])
nb_pts = np.array([2, 5, 10])

res = np.zeros((nb_pts.size, nb_intervals.size))
err = np.zeros((nb_pts.size, nb_intervals.size))

for i_n, n in enumerate(nb_pts):
    xk, wk = coeffs_clenshawcurtis(n-1)
    for i_m, m in enumerate(nb_intervals):
        x = np.linspace(xmin, xmax, m+1)
        for j in range(m):
            xminj = x[j]
            xmaxj = x[j+1]
            xkj = xminj + ((xmaxj-xminj)/2)*(xk+1) 
            wkj = ((xmaxj-xminj)/2)*wk
            res[i_n, i_m] += np.sum(wkj*f(xkj))
        err[i_n, i_m] = np.abs(res[i_n, i_m]-res_exa) /  np.abs(res_exa)

**Error as a function of the number of quadrature points**

For subdivisions into different numbers of sub-intervals, we plot the error as a function of the number of quadrature points:

In [ ]:
fig01 = go.Figure()
for i_m, m in enumerate(nb_intervals):
    fig01.add_trace(go.Scatter(x=nb_pts, y=err[:,i_m], mode="markers", name=f"m={m}"))
fig01.update_xaxes(type="log", exponentformat='e', title_text='Nb. of quadrature points per interval')
fig01.update_yaxes(type="log", exponentformat='e', title_text='Error')
fig01.update_layout(height=500, title="Error as a function of the number of quadrature points", legend_title_text="Nb. of intervals")
fig01.show()

**Error as a function of the number of intervals**

For different numbers of quadrature points $n$, we plot the error as a function of the number of intervals:

In [ ]:
fig02 = go.Figure()
for i_n, n in enumerate(nb_pts):
    fig02.add_trace(go.Scatter(x=nb_intervals, y=err[i_n], mode="markers", name=f"n={n}"))
fig02.update_xaxes(type="log", exponentformat='e', title_text="Nb. of intervals")
fig02.update_yaxes(type="log", exponentformat='e', title_text='Error')
fig02.update_layout(height=500, title="Error as a function of the number of intervals", legend_title_text='Nb. of quadrature points')
fig02.show()